# Actividad Preguntas Taxi de NY

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("ActividadTaxiNY")
    .master("local[4]")                       # 4 núcleos
    .config("spark.driver.memory", "8g")       # 8 GB para el driver
    .config("spark.executor.memory", "4g")     # 4 GB para ejecutores
    .config("spark.sql.shuffle.partitions", "8")  # Particiones para shuffles
    .config("spark.sql.adaptive.enabled", "true") # Ejecución adaptativa (AQE)
    .getOrCreate()
)


In [6]:
df = spark.read.parquet("yellow_tripdata_2017-10.parquet")

**¿Cuántas columnas tiene el DataFrame?**

In [11]:
print(f"Columnas: {len(df.columns)}")
df.printSchema()

Columnas: 19
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: void (nullable = true)
 |-- airport_fee: void (nullable = true)



**¿Cuál es el tipo de dato de la columna Fare_amont?**

In [10]:
print("Tipo de dato de la columna fare_amount:", df.schema["fare_amount"].dataType)

Tipo de dato de la columna fare_amount: DoubleType()


**Muestra las últimas 3 filas del DataFrame usando .tail(3)**

In [14]:
# Obtener las últimas 3 filas
ultimas_3 = df.tail(3)

# Mostrar cada registro
for i, fila in enumerate(ultimas_3, start=1):
    print(f"\nRegistro {i}:")
    print(fila)


Registro 1:
Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2017, 10, 31, 23, 33, 17), tpep_dropoff_datetime=datetime.datetime(2017, 11, 1, 0, 2, 51), passenger_count=2, trip_distance=6.2, RatecodeID=1, store_and_fwd_flag='N', PULocationID=100, DOLocationID=255, payment_type=1, fare_amount=24.0, extra=0.5, mta_tax=0.5, tip_amount=2.0, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=27.3, congestion_surcharge=None, airport_fee=None)

Registro 2:
Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2017, 10, 31, 23, 25, 32), tpep_dropoff_datetime=datetime.datetime(2017, 10, 31, 23, 31, 22), passenger_count=1, trip_distance=0.82, RatecodeID=1, store_and_fwd_flag='N', PULocationID=264, DOLocationID=264, payment_type=2, fare_amount=5.5, extra=0.5, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=6.8, congestion_surcharge=None, airport_fee=None)

Registro 3:
Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2017, 10, 31, 23, 34, 

**¿Cuántos viajes tienen passenger_count nulo o 0?**

In [17]:
from pyspark.sql.functions import col, sum as spark_sum

df.select(
    spark_sum(
        ((col("passenger_count").isNull()) | (col("passenger_count") == 0))
        .cast("int")
    ).alias("viajes_nulos_o_cero")
).show()

+-------------------+
|viajes_nulos_o_cero|
+-------------------+
|              36311|
+-------------------+

